In [ ]:
%%bash
apt-get -qq install -y mafft iqtree > /dev/null
pip -q install ete3
pip -q install biopython
which mafft; which iqtree2 || which iqtree

In [ ]:
import shutil, zipfile
from google.colab import files
from Bio import Phylo

In [ ]:
# Pack folders with deduplicated fastas (felidae_dedup, canidae_dedup),
# the outgroup fastas (hyena, bear), and export_for_mafft.py into fastas_pack.zip
up = files.upload()          # choose fastas_pack.zip
!unzip -oq fastas_pack.zip
!mv hyena.fasta Crocuta_crocuta_complete.fasta
!mv bear.fasta Ursus_arctos_complete.fasta
!ls

In [ ]:
%%bash
python export_for_mafft.py felidae_dedup felidae_mt \
    --anchor-file felidae_dedup/Pardofelis_marmorata_complete.fasta \
    --anchor-accession MW257207.1 --anchor-start 0 \
    --outgroup Crocuta_crocuta_complete.fasta
python export_for_mafft.py canidae_dedup canidae28_mt \
    --anchor-file canidae_dedup/Canis_familiaris_complete.fasta \
    --anchor-accession NC_002008.4 --anchor-start 0 \
    --drop Lupulella_adusta --outgroup Ursus_arctos_complete.fasta
cat felidae_mt_records.tsv canidae28_mt_records.tsv | cut -f1,2,5,9
mafft --auto --thread 2 felidae_mt.fasta > felidae_mt_aln.fasta
mafft --auto --thread 2 canidae28_mt.fasta > canidae28_mt_aln.fasta

In [ ]:
!iqtree2 -s felidae_mt_aln.fasta -m GTR+G -bb 1000 -nt AUTO -seed 1 -o Crocuta_crocuta

In [ ]:
!iqtree2 -s canidae28_mt_aln.fasta -m GTR+G -bb 1000 -nt AUTO -seed 1 -o Ursus_arctos

In [ ]:
def strip(treefile, outgroup, out):
    t = Phylo.read(treefile, "newick")
    t.root_with_outgroup(outgroup)
    t.prune(outgroup)
    Phylo.write(t, out, "newick", plain=True)     # topology only
    print(out, t.count_terminals(), "leaves")

In [ ]:
strip("felidae_mt_aln.fasta.treefile", "Crocuta_crocuta", "felidae_mt_ml.nwk")
strip("canidae28_mt_aln.fasta.treefile", "Ursus_arctos", "canidae28_mt_ml.nwk")

for fam in ["felidae", "canidae28"]:
    # copy the IQ-TREE tree under the repository name
    shutil.copy(f"{fam}_mt_aln.fasta.treefile", f"{fam}_mt_ml_support.nwk")
    members = [
        f"{fam}_mt_aln.fasta",          # alignment
        f"{fam}_mt_aln.fasta.iqtree",   # report
        f"{fam}_mt_aln.fasta.log",      # command, seed, composition test
        f"{fam}_mt_ml_support.nwk",     # tree with bootstrap support and outgroup
        f"{fam}_mt_ml.nwk",             # clean tree, outgroup removed
        f"{fam}_mt_records.tsv",        # records, rotation offsets
    ]
    with zipfile.ZipFile(f"{fam}_ml.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for m in members:
            z.write(m)
    files.download(f"{fam}_ml.zip")